# Lesson 3 - Building a QA Agent with Vertex AI


In this lesson, you will build a basic Question Answering (QA) agent using a lightweight Google Gemini model on Vertex AI. You will start by directly interacting with the model to analyze a PDF document containing health insurance policy details. Then, you will refactor this logic into a reusable, configurable Python class that can use either the Google backend or the original Anthropic Claude backend, laying the groundwork for the next lesson where you will wrap this agent in an Agent2Agent (A2A) server.


<p style="background-color:#fff6ff; padding:15px; border-width:3px; border-color:#efe6ef; border-style:solid; border-radius:6px"> 💻 &nbsp; <b>To Access the <code>requirements.txt</code> and <code>helpers</code> files, and the <code>data</code> folder:</b> 1) click on the <em>"File"</em> option on the top menu of the notebook and then 2) click on <em>"Open"</em>. The PDF document is in the <code>data</code> folder.</p> 

## 3.1. Import Libraries and Setup

First, import the necessary libraries. You will use the Google GenAI SDK to call a lightweight Gemini model through Vertex AI and standard libraries to load configuration and the PDF document.


In [8]:
import os
from pathlib import Path

from IPython.display import Markdown, display
from dotenv import load_dotenv
from google import genai
from google.genai import types

from helpers import authenticate


In [9]:
_ = load_dotenv()

You can see an example of the environment variables defined [here](https://github.com/holtskinner/A2AWalkthrough/blob/main/example.env).

## 3.2. Initialize the Client and Load Data

Here you initialize a Vertex AI Gemini client using your local Google Cloud credentials. The model defaults to `gemini-2.5-flash-lite`, which avoids the Claude partner-model quota while you wait for the quota increase.


In [10]:
load_dotenv()

policy_location = os.getenv("POLICY_GOOGLE_VERTEX_LOCATION") or os.getenv(
    "GOOGLE_CLOUD_LOCATION",
    "global",
)
policy_model = os.getenv("POLICY_GOOGLE_VERTEX_MODEL", "gemini-2.5-flash-lite")
credentials, project_id = authenticate(location=policy_location)


**Note:** When using models through Vertex AI, there is more than one way to authenticate. This local version uses Application Default Credentials, so run `gcloud auth application-default login` if credentials are missing. To switch the reusable agent back to Claude later, set `POLICY_AGENT_BACKEND=anthropic` after your Claude quota is approved.


In [11]:
client = genai.Client(
    vertexai=True,
    credentials=credentials,
    project=project_id,
    location=policy_location,
)


You also read the insurance policy PDF (`2026AnthemgHIPSBC.pdf`) as bytes so it can be passed to Gemini as inline document context. You can find the PDF document in the [course repo](https://github.com/holtskinner/A2AWalkthrough/tree/main/data) or in the `data` folder.


In [12]:
pdf_data = Path("../data/2026AnthemgHIPSBC.pdf").read_bytes()


## 3.3. Query the Model

Now you will send a specific query to the model: "How much would I pay for mental health therapy?". You provide the model with a system instruction to act as an expert insurance agent and pass the PDF document alongside the user's text prompt.

In [13]:
prompt = "How much would I pay for mental health therapy?"

In [14]:
POLICY_SYSTEM_PROMPT = """You are an expert insurance agent designed to assist with
coverage queries. Use the provided documents to answer questions
about insurance policies. If the information is not available in
the documents, respond with 'I don't know'"""

response = client.models.generate_content(
    model=policy_model,
    contents=[
        types.Part.from_bytes(
            data=pdf_data,
            mime_type="application/pdf",
        ),
        prompt,
    ],
    config=types.GenerateContentConfig(
        system_instruction=POLICY_SYSTEM_PROMPT,
        max_output_tokens=1024,
        temperature=0.1,
    ),
)


In [15]:
response_text = response.text.replace("$", r"\$")
display(Markdown(response_text))


If you need mental health, behavioral health, or substance abuse services, you will pay 10% coinsurance for outpatient services and inpatient services if you use an in-network provider. If you use an out-of-network provider, you will pay 30% coinsurance for outpatient services and inpatient services.

## 3.4. Refactor into an Agent Class

To make this code reusable and easier to integrate into an A2A server later, you will wrap the logic into a `PolicyAgent` class in a file named `agents.py`. The public class stays simple, while the implementation delegates to a configurable backend selected by `POLICY_AGENT_BACKEND`: `google` for Gemini on Vertex AI or `anthropic` for Claude on Vertex AI.


In [16]:
%%writefile agents.py
import base64
import os
from pathlib import Path

from anthropic import AnthropicVertex
from anthropic.types import (
    Base64PDFSourceParam,
    DocumentBlockParam,
    MessageParam,
    TextBlockParam,
)
from google import genai
from google.genai import types

from helpers import authenticate

PROJECT_ROOT = Path(__file__).resolve().parents[1]
POLICY_SYSTEM_PROMPT = (
    "You are an expert insurance agent designed to assist with coverage queries. "
    "Use the provided documents to answer questions about insurance policies. "
    "If the information is not available in the documents, respond with "
    "'I don't know'"
)


def policy_document_path() -> Path:
    return Path(
        os.getenv(
            "POLICY_DOCUMENT_PATH",
            str(PROJECT_ROOT / "data" / "2026AnthemgHIPSBC.pdf"),
        )
    ).expanduser()


def escape_markdown_dollars(text: str) -> str:
    return text.replace("$", r"\$")


class PolicyAgent:
    def __init__(self) -> None:
        backend = os.getenv("POLICY_AGENT_BACKEND", "google").strip().lower()
        if backend == "google":
            self.backend = GoogleVertexPolicyBackend()
        elif backend == "anthropic":
            self.backend = AnthropicVertexPolicyBackend()
        else:
            raise ValueError(
                "Unsupported POLICY_AGENT_BACKEND. Use 'google' or 'anthropic'."
            )

    def answer_query(self, prompt: str) -> str:
        return self.backend.answer_query(prompt)


class GoogleVertexPolicyBackend:
    def __init__(self) -> None:
        location = os.getenv("POLICY_GOOGLE_VERTEX_LOCATION") or os.getenv(
            "GOOGLE_CLOUD_LOCATION",
            "global",
        )
        credentials, project_id = authenticate(location=location)
        self.client = genai.Client(
            vertexai=True,
            credentials=credentials,
            project=project_id,
            location=location,
        )
        self.model = os.getenv("POLICY_GOOGLE_VERTEX_MODEL", "gemini-2.5-flash-lite")
        self.pdf_data = policy_document_path().read_bytes()

    def answer_query(self, prompt: str) -> str:
        response = self.client.models.generate_content(
            model=self.model,
            contents=[
                types.Part.from_bytes(
                    data=self.pdf_data,
                    mime_type="application/pdf",
                ),
                prompt,
            ],
            config=types.GenerateContentConfig(
                system_instruction=POLICY_SYSTEM_PROMPT,
                max_output_tokens=1024,
                temperature=0.1,
            ),
        )
        if not response.text:
            raise RuntimeError("The Google Vertex model returned an empty response.")
        return escape_markdown_dollars(response.text)


class AnthropicVertexPolicyBackend:
    def __init__(self) -> None:
        location = os.getenv("ANTHROPIC_VERTEX_LOCATION", "global")
        credentials, project_id = authenticate(location=location)
        self.client = AnthropicVertex(
            project_id=project_id,
            region=location,
            credentials=credentials,
        )
        with policy_document_path().open("rb") as file:
            self.pdf_data = base64.standard_b64encode(file.read()).decode("utf-8")

    def answer_query(self, prompt: str) -> str:
        response = self.client.messages.create(
            model=os.getenv("ANTHROPIC_VERTEX_MODEL", "claude-haiku-4-5@20251001"),
            max_tokens=1024,
            system=POLICY_SYSTEM_PROMPT,
            messages=[
                MessageParam(
                    role="user",
                    content=[
                        DocumentBlockParam(
                            type="document",
                            source=Base64PDFSourceParam(
                                type="base64",
                                media_type="application/pdf",
                                data=self.pdf_data,
                            ),
                        ),
                        TextBlockParam(
                            type="text",
                            text=prompt,
                        ),
                    ],
                )
            ],
        )
        return escape_markdown_dollars(response.content[0].text)


Overwriting agents.py


## 3.5. Test the Agent Class

Finally, import the `PolicyAgent` class you just created and test it with the same query. With the default configuration, this uses the Google Vertex backend.


In [17]:
from agents import PolicyAgent

print("Running Health Insurance Policy Agent")
agent = PolicyAgent()
prompt = "How much would I pay for mental health therapy?"

response = agent.answer_query(prompt)
display(Markdown(response))


Running Health Insurance Policy Agent


If you need mental health, behavioral health, or substance abuse services, you will pay 10% coinsurance for outpatient services and inpatient services if you use an in-network provider. If you use an out-of-network provider, you will pay 30% coinsurance for outpatient services and inpatient services.

## 3.6. Resources

- [Gemini on Vertex AI](https://cloud.google.com/vertex-ai/generative-ai/docs/model-reference/inference)
- [Anthropic on Vertex AI](https://docs.cloud.google.com/vertex-ai/generative-ai/docs/partner-models/claude)
- [Course Repo](https://github.com/holtskinner/A2AWalkthrough)


<div style="background-color:#fff6ff; padding:13px; border-width:3px; border-color:#efe6ef; border-style:solid; border-radius:6px">
<p> ⬇ &nbsp; <b>Download Notebooks:</b> 1) click on the <em>"File"</em> option on the top menu of the notebook and then 2) click on <em>"Download"</em>.</p>
</div>
